# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration of loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The metadata object provides metadata fields as attributes
meta = dataset.metadata
print(f"Dataset title: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}\n")
print(f"Published: {meta.datePublished}\n")

## 2. Data Overview
Review available record sets, their field `@id`s and field names. All entities are referenced by their `@id` as required by the Croissant specification and best interoperability practices.

In [ ]:
recordset_list = list(dataset.record_sets())
print(f"Number of RecordSets in this dataset: {len(recordset_list)}\n")
for rs in recordset_list:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"   - {field.name} (id={field.id}, type={field.data_type})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. For each record set, we extract records and construct a pandas DataFrame using the record set `@id` as the reference key.

In [ ]:
dataframes = {}
for rs in recordset_list:
    recs = list(dataset.records(record_set=rs.id))  # Reference record set by `@id`
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} rows for RecordSet '{rs.name}' (@id={rs.id})")
        print("Columns:", list(df.columns))
        print()
    else:
        print(f"No records found for RecordSet '{rs.name}' (@id={rs.id})\n")
# Pick the first record set as main for demo
if len(dataframes) > 0:
    main_rs_id = next(iter(dataframes))
    print(f"Using RecordSet @id: {main_rs_id} as primary for further analysis.")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic EDA: filtering a numeric field, normalization, and grouping. All field references are through their Croissant `@id`.

Select a numeric field (e.g., GAD-7 Score or age) and a grouping field (e.g., gender or education level) from the field overview above.

In [ ]:
# Please update `numeric_field_id` and `group_field_id` below to valid croissant field @id references for your main record set.

# Example: Suppose your main record set is surveys, with fields for age, gad7_score, and gender.
numeric_field_id = None
group_field_id = None
# Pick the most relevant (for demonstration, use the first numeric field found)

df = dataframes[main_rs_id]
recordset = [r for r in recordset_list if r.id == main_rs_id][0]
numeric_field_candidates = [f for f in recordset.fields if f.data_type in ('Integer', 'Float', 'Number')]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0].id  # Use `@id`
else:
    print("No numeric field available.")
    numeric_field_id = df.columns[0]  # fallback
group_field_candidates = [f for f in recordset.fields if f.data_type in ('Text', 'String') and f.id != numeric_field_id]
if group_field_candidates:
    group_field_id = group_field_candidates[0].id
else:
    group_field_id = df.columns[0]  # fallback

# Check field ids and types
print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}\n")

# Demonstrate basic EDA
threshold = df[numeric_field_id].mean()  # use mean as threshold for example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records\n")

# Normalize numeric field
filtered_df[numeric_field_id + '_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"First 5 normalized values:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by the group field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Group means by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib and seaborn for simple visualizations.

Below, we plot a histogram for the selected numeric field and a boxplot by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field by group field (if group is categorical)
if group_field_id in df.columns and df[group_field_id].nunique() < 20:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We've demonstrated how to load a Croissant dataset using `mlcroissant`, inspect available record sets and fields, extract data by referencing their unique `@id`s, and perform basic data analysis and EDA.

* Make sure to refer to Croissant element documentation, which provides both `@id` and field types to facilitate reliable, interoperable workflows.

Further steps can include advanced visualizations, machine learning model fitting, and joining across record sets, always using Croissant's `@id` conventions.